# Лабораторная работа №2. Метод ближайших соседей

**Дисциплина:** «Интеллектуальные методы анализа данных»
**Тема:** реализация метода классификации данных с использованием природных
временных рядов
**Вариант 14:** нейтронный монитор Алма-Ата (AATB), 2–10 мая 2017 г.
**Студент гр. 3381 Щеглов М.А.**

**Цель работы** — построение классификатора данных геофизического мониторинга
на примере наблюдений нейтронных мониторов.

Порядок работы: скачиваем одноминутный ряд с [nmdb.eu/nest](https://www.nmdb.eu/nest/),
заменяем пропуски и выбросы медианой, нарезаем ряд на векторы-истории длиной
*m* = 180, ищем для последнего (тестового) вектора ближайших соседей по трём
метрикам, усредняем то, что следовало за соседями, и сравниваем прогноз с
фактическими значениями следующего часа.

## 1. Настройка

In [ ]:
%matplotlib inline

import os
import re
import urllib.parse
import urllib.request
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

if Path("lab2").is_dir():          # если ноутбук запущен из корня репозитория
    os.chdir("lab2")

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#e4e3df"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

: 

In [ ]:
STATION = "AATB"                                  # станция по варианту
START, END = "2017-05-02 00:00", "2017-05-10 23:59"            # период
FACT_START, FACT_END = "2017-05-11 00:00", "2017-05-11 00:59"  # час проверки

M = 180           # длина вектора-истории, отсчётов
HORIZON = 60      # на сколько минут вперёд строим прогноз
KS = (2, 3, 4)    # число ближайших соседей
SIGMA = 3         # порог выбросов (правило трёх сигм)

DATA = Path("data")
FIGS = Path("figures")
FIGS.mkdir(exist_ok=True)

COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]   # цвета для трёх метрик
DARK, GREY = "#1f1f1f", "#8a8985"

## 2. Загрузка данных

Данные берём из базы NMDB через интерфейс NEST: одноминутные значения скорости
счёта, скорректированные на давление и эффективность детектора (RCORR_E).
Сервер отдаёт HTML-страницу, внутри которой в блоке `<pre><code>` лежат строки
вида «дата время;значение».

Кроме периода по варианту скачиваем ещё 60 значений сразу после него
(11.05.2017 00:00–00:59) — это «будущее», с которым сравним прогноз.

In [ ]:
def nmdb(start, end, filename):
    """Скачивает (если файла ещё нет) и читает одноминутный ряд RCORR_E."""
    path = DATA / filename
    if not path.exists():
        t0, t1 = pd.Timestamp(start), pd.Timestamp(end)
        query = urllib.parse.urlencode({
            "formchk": 1, "stations[]": STATION, "tabchoice": "revori",
            "dtype": "corr_for_efficiency", "tresolution": 0, "yunits": 0,
            "date_choice": "bydate", "output": "ascii",
            "start_day": t0.day, "start_month": t0.month, "start_year": t0.year,
            "start_hour": t0.hour, "start_min": t0.minute,
            "end_day": t1.day, "end_month": t1.month, "end_year": t1.year,
            "end_hour": t1.hour, "end_min": t1.minute,
        })
        path.parent.mkdir(exist_ok=True)
        print("скачиваю", path.name)
        url = "https://www.nmdb.eu/nest/draw_graph.php?" + query
        path.write_bytes(urllib.request.urlopen(url, timeout=120).read())

    body = re.search(r"<pre><code>(.*?)</code></pre>",
                     path.read_text(encoding="utf-8"), re.S).group(1)
    rows = [line.split(";") for line in body.splitlines()
            if re.match(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}", line)]
    return pd.Series({pd.Timestamp(t): float(v) for t, v in rows},
                     name="RCORR_E")

In [ ]:
raw = nmdb(START, END, "aatb_history.html")
fact_raw = nmdb(FACT_START, FACT_END, "aatb_actual.html")

print(f"исторический ряд: {len(raw)} значений, "
      f"{raw.index[0]:%d.%m %H:%M} — {raw.index[-1]:%d.%m %H:%M}")
print(f"фактический час:  {len(fact_raw)} значений")
raw.head()

## 3. Пропуски и выбросы

Сервер просто не присылает строки за те минуты, когда данных нет. Поэтому
сначала «натягиваем» ряд на равномерную минутную сетку — недостающие отсчёты
сами становятся пропусками.

In [ ]:
grid = pd.date_range(START, END, freq="min")
series = raw.reindex(grid)
missing = series.isna()

print(f"ожидалось {len(grid)} отсчётов, получено {len(raw)}")
print(f"пропусков: {missing.sum()} ({missing.mean():.2%})")
series[missing].index[:8]

Выбросы ищем по правилу трёх сигм — всё, что вышло за μ ± 3σ.

In [ ]:
mean, std = series.mean(), series.std()
low, high = mean - SIGMA * std, mean + SIGMA * std
outliers = (series < low) | (series > high)

print(f"μ = {mean:.2f}, σ = {std:.2f}, границы нормы [{low:.2f}; {high:.2f}]")
print(f"выбросов: {outliers.sum()} ({outliers.mean():.2%})")
series[outliers].sort_values(ascending=False).head()

Восемь самых больших значений идут подряд 8 мая с 08:55 до 11:17 — это не
геофизика, а аппаратные всплески. Заменяем и пропуски, и выбросы медианой
оставшихся («чистых») значений — это аналог функции `median` из задания.

In [ ]:
median = series[~missing & ~outliers].median()
clean = series.mask(missing | outliers, median)

print(f"заменено {int((missing | outliers).sum())} значений "
      f"медианой {median:.2f} имп./с")

То же самое для часа фактических значений, с которым будем сравнивать прогноз.

In [ ]:
fact_grid = pd.date_range(FACT_START, FACT_END, freq="min")
fact = fact_raw.reindex(fact_grid)

f_missing = fact.isna()
f_mean, f_std = fact.mean(), fact.std()
f_outliers = (fact < f_mean - SIGMA * f_std) | (fact > f_mean + SIGMA * f_std)
f_median = fact[~f_missing & ~f_outliers].median()
fact_clean = fact.mask(f_missing | f_outliers, f_median)

print(f"пропусков {f_missing.sum()}, выбросов {f_outliers.sum()}, "
      f"медиана {f_median:.2f} имп./с")

**Таблица 1** — итоги предварительной обработки.

In [ ]:
def summary(grid, raw, missing, mean, std, outliers, median):
    """Строка итогов предварительной обработки для одного ряда."""
    return [f"{len(grid)}", f"{len(raw)}",
            f"{missing.sum()} ({missing.mean():.2%})",
            f"{mean:.2f}", f"{std:.2f}",
            f"[{mean - SIGMA * std:.2f}; {mean + SIGMA * std:.2f}]",
            f"{outliers.sum()} ({outliers.mean():.2%})", f"{median:.2f}"]


table1 = pd.DataFrame({
    "Исторические данные": summary(grid, raw, missing, mean, std, outliers,
                                   median),
    "Фактические значения": summary(fact_grid, fact_raw, f_missing, f_mean,
                                    f_std, f_outliers, f_median),
}, index=["Ожидаемое число отсчётов", "Получено отсчётов", "Пропусков",
          "Среднее μ, имп./с", "СКО σ, имп./с", "Границы нормы μ ± 3σ",
          "Выбросов", "Медиана для замены, имп./с"])
table1

### Рис. 1 — ряд до и после обработки

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(series.index, series, color=COLORS[0], linewidth=0.6)
axes[0].scatter(series.index[outliers], series[outliers], s=18,
                color=COLORS[1], zorder=3, label=f"выбросы ({outliers.sum()})")
axes[0].scatter(series.index[missing], np.full(missing.sum(), series.min() - 5),
                s=18, marker="|", color=DARK, zorder=3,
                label=f"пропуски ({missing.sum()})")
axes[0].set_title("а) исходный ряд")

hourly = clean.resample("60min").mean()
axes[1].plot(clean.index, clean, color=COLORS[0], linewidth=0.6,
             label="одноминутные значения")
axes[1].plot(hourly.index + pd.Timedelta(minutes=30), hourly, color=DARK,
             linewidth=2, label="среднее за час")
axes[1].set_title("б) после замены пропусков и выбросов медианой")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
axes[1].set_xlabel("дата (UTC), 2017 г.")

for ax in axes:
    ax.set_ylabel("RCORR_E, имп./с")
    ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGS / "fig1_series.png", dpi=200)

Среднечасовой уровень меняется медленно — от 1408 до 1436 имп./с за девять
дней, то есть примерно на 2 %. Всё остальное на минутном масштабе — случайные
флуктуации счёта.

## 4. Векторы-истории

Тестовый вектор — последние 180 значений ряда. Векторы-истории — все отрезки
длины 180, которые получаются скользящим окном с шагом в один отсчёт и целиком
лежат до тестового вектора. Для каждого запоминаем 60 следующих за ним
значений: именно они и усредняются при прогнозе.

In [ ]:
x = clean.to_numpy(dtype=float)
test = x[-M:]                                       # тестовый вектор
n = len(x) - 2 * M + 1                              # сколько историй помещается
histories = sliding_window_view(x, M)[:n]           # прошлые отрезки
futures = sliding_window_view(x, HORIZON)[M:M + n]  # что было после каждого

print(f"длина ряда: {len(x)}, векторов-историй: {n}")
print(f"тестовый вектор: {clean.index[-M]:%d.%m.%Y %H:%M}"
      f"–{clean.index[-1]:%H:%M}, среднее {test.mean():.2f} имп./с")
histories.shape, futures.shape

## 5. Поиск ближайших соседей

Для тестового вектора $x$ и каждого вектора-истории $y$ считаем три расстояния:

$$d_E = \sqrt{\sum_{j=1}^{m}(x_j - y_j)^2}, \qquad
  d_M = \sum_{j=1}^{m}|x_j - y_j|, \qquad
  d_Ч = \max_{j=1..m}|x_j - y_j|$$

Считаем сразу для всей матрицы историй, затем сортируем по возрастанию
расстояния и берём 4 ближайших (при *k* = 2 и 3 используем первые 2 и 3).

In [ ]:
diff = np.abs(histories - test)
dist = {
    "Евклидова": np.sqrt((diff ** 2).sum(axis=1)),
    "Манхэттенская": diff.sum(axis=1),
    "Чебышева": diff.max(axis=1),
}
neighbors = {name: np.argsort(d, kind="stable")[:max(KS)]
             for name, d in dist.items()}

for name, idx in neighbors.items():
    print(f"{name:<14} ближайшие индексы: {idx.tolist()}")

**Таблица 2** — найденные соседи (индекс = номер первого отсчёта отрезка).

In [ ]:
table2 = pd.DataFrame([
    {"Метрика": name, "№": rank, "Индекс": int(i),
     "Интервал (UTC)": f"{clean.index[i]:%d.%m %H:%M}"
                       f"–{clean.index[i + M - 1]:%H:%M}",
     "Расстояние": round(float(dist[name][i]), 3)}
    for name, idx in neighbors.items()
    for rank, i in enumerate(idx, 1)
])
table2

Проверим, чем на самом деле определяется расстояние: посчитаем корреляцию
расстояния с модулем разности средних уровней тестового вектора и
вектора-истории по всем 12601 отрезкам.

In [ ]:
level = np.abs(histories.mean(axis=1) - test.mean())
for name, d in dist.items():
    print(f"{name:<14} корреляция с разностью уровней: "
          f"{np.corrcoef(d, level)[0, 1]:.2f}")

У евклидовой и манхэттенской метрик корреляция 0,85–0,86: они фактически
отбирают отрезки с таким же средним уровнем счёта. Метрика Чебышева (0,39)
определяется одним наибольшим отклонением, то есть случайным всплеском шума.

### Рис. 2 — тестовый вектор и его соседи

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, sharey=True)
steps = np.arange(-M, 0)

for ax, (name, idx), color in zip(axes, neighbors.items(), COLORS):
    for j, i in enumerate(idx):
        ax.plot(steps, histories[i], color=color, linewidth=0.8, alpha=0.55,
                label=f"{len(idx)} ближайших соседа" if j == 0 else None)
    ax.plot(steps, test, color=DARK, linewidth=1.6, label="тестовый вектор")
    ax.set_title(f"метрика: {name}")
    ax.set_ylabel("RCORR_E, имп./с")
    ax.legend(loc="upper right")

axes[-1].set_xlabel("номер отсчёта относительно конца вектора, мин")
fig.tight_layout()
fig.savefig(FIGS / "fig2_neighbors.png", dpi=200)

Соседи не повторяют форму тестового вектора по точкам — совпадают только
средний уровень и размах колебаний.

## 6. Прогноз по соседям

Прогноз — простое среднее значений, следующих за *k* ближайшими соседями:

$$\hat{y}_t = \frac{1}{k}\sum_{s=1}^{k} x(i_s + m + t), \qquad t = 0, \ldots, 59$$

Для сравнения возьмём наивный прогноз — среднее тестового вектора, постоянное
на всём горизонте.

In [ ]:
def forecast(name, k):
    """Среднее того, что следовало за k ближайшими соседями по метрике."""
    idx = np.argsort(dist[name], kind="stable")[:k]
    return futures[idx].mean(axis=0)


y = fact_clean.to_numpy(dtype=float)        # фактические 60 значений
naive = np.full(HORIZON, test.mean())       # наивный прогноз
preds = {(name, k): forecast(name, k) for name in dist for k in KS}

print(f"наивный прогноз: {test.mean():.2f} имп./с")
print(f"СКО фактических значений: {y.std():.2f} имп./с")

**Таблица 3** — ошибки прогноза на 60 мин.

In [ ]:
def errors(pred):
    """MAE, RMSE и MAPE прогноза относительно фактических значений."""
    d = y - pred
    return {"MAE, имп./с": np.mean(np.abs(d)),
            "RMSE, имп./с": np.sqrt(np.mean(d ** 2)),
            "MAPE, %": np.mean(np.abs(d) / y) * 100}


table3 = pd.DataFrame([{"Метрика": name, "k": k, **errors(pred)}
                       for (name, k), pred in preds.items()])
table3.loc[len(table3)] = {"Метрика": "Среднее тестового вектора", "k": "—",
                           **errors(naive)}
table3.round(3)

### Рис. 3–5 — прогноз по каждой метрике

In [ ]:
files = {"Евклидова": "euclid", "Манхэттенская": "manhattan",
         "Чебышева": "chebyshev"}

for number, name in enumerate(dist, start=3):
    fig, ax = plt.subplots()
    ax.plot(fact_clean.index, y, color=DARK, linewidth=2,
            label="фактические значения")
    for k, color in zip(KS, COLORS):
        ax.plot(fact_clean.index, preds[name, k], color=color, linewidth=1.6,
                label=f"прогноз, k = {k}")
    ax.axhline(test.mean(), color=GREY, linewidth=1.4, linestyle="--",
               label="среднее тестового вектора")
    ax.set_title(f"прогноз на {HORIZON} мин — метрика: {name}")
    ax.set_ylabel("RCORR_E, имп./с")
    ax.set_xlabel("время (UTC), 11.05.2017")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=5,
              frameon=False)
    fig.tight_layout()
    fig.savefig(FIGS / f"fig{number}_forecast_{files[name]}.png", dpi=200)

### Рис. 6 — тестовый вектор, факт и прогнозы при k = 4

In [ ]:
fig, ax = plt.subplots()

ax.plot(clean.index[-M:], test, color=GREY, linewidth=0.9,
        label="тестовый вектор (последние 180 значений)")
ax.plot(fact_clean.index, y, color=DARK, linewidth=1.6,
        label="фактические значения")
for name, color in zip(dist, COLORS):
    ax.plot(fact_clean.index, preds[name, 4], color=color, linewidth=1.6,
            label=f"прогноз — {name}, k = 4")

ax.axvline(fact_clean.index[0], color=DARK, linewidth=1, linestyle=":")
ax.set_title("тестовый вектор и прогноз следующих 60 значений")
ax.set_ylabel("RCORR_E, имп./с")
ax.set_xlabel("время (UTC), 10–11.05.2017")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3,
          frameon=False)
fig.tight_layout()
fig.savefig(FIGS / "fig6_overview.png", dpi=200)

## 7. Как ошибка зависит от числа соседей

В задании k = 2–4, но интересно посмотреть шире: посчитаем MAE для k от 1 до 20.

In [ ]:
ks = range(1, 21)
scan = {name: [errors(forecast(name, k))["MAE, имп./с"] for k in ks]
        for name in dist}

fig, ax = plt.subplots()
ax.axvspan(min(KS) - 0.5, max(KS) + 0.5, color="#f1f0ec", label="k по заданию")
for (name, values), color in zip(scan.items(), COLORS):
    ax.plot(list(ks), values, color=color, linewidth=2, marker="o",
            markersize=5, label=name)
ax.axhline(errors(naive)["MAE, имп./с"], color=GREY, linewidth=1.4,
           linestyle="--", label="среднее тестового вектора")

ax.set_title("зависимость MAE прогноза от числа соседей k")
ax.set_xlabel("число соседей k")
ax.set_ylabel("MAE, имп./с")
ax.set_xticks(list(ks))
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGS / "fig7_k_scan.png", dpi=200)

## Выводы

Прогноз одноминутного ряда нейтронного монитора Алма-Ата B построен методом
ближайших соседей на векторах-историях длины *m* = 180. Метод работает как
классификатор: тестовый вектор относится к группе наиболее похожих отрезков
истории, а прогноз строится по тому, что следовало за ними.

При подготовке данных 38 пропусков (0,29 %) и 29 выбросов (0,22 %), включая
серию аппаратных всплесков 8 мая, заменены медианой 1422,5 имп./с. Из
обработанного ряда сформирован 12601 вектор-история.

Ошибка прогноза MAE составила от 8,99 до 12,47 имп./с при СКО фактических
значений 10,60 имп./с и MAE наивного прогноза 9,21 имп./с. Лучший вариант —
манхэттенская метрика при *k* = 3; хуже всего работает метрика Чебышева,
потому что определяется одной наибольшей разностью, то есть случайным
выбросом шума. С ростом k прогноз сглаживается и ошибка падает (рис. 7).

Основная причина низкой точности — свойства самих данных: на минутном
масштабе ряд состоит в основном из случайных флуктуаций счёта (СКО около
11 имп./с), а медленные вариации за девять дней составляют лишь около 2 %.
Поэтому отдельные будущие значения по прошлым 180 отсчётам предсказать
нельзя: метрики отбирают отрезки с близким средним уровнем, и при больших k
прогноз сводится к оценке этого уровня.